# 🌾 Minería final Gravetal — modo CSV (sin Snowflake)
F1 se recomputa aquí (1,897 filas en repo). F2–F4 grafican los artefactos versionados en `data/` (genéralos con los scripts y commítéalos si faltan). Requiere `requirements.txt` sin snowflake.

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
def need(path):
    print(('OK ' if os.path.exists(path) else 'FALTA ') + path)
    return os.path.exists(path)
for f in ['data/productores_features_v2.csv', 'data/alerta_top.csv', 'data/riesgo_silo.csv',
          'data/iot_sensor_regimen.csv', 'data/mercado_regimen_hora.csv', 'data/asignacion_optima.csv']:
    need(f)

## Frente 1 · Segmentos (recomputo completo)
K-means + DBSCAN sobre 16 features con dispersión. Cuarentena 525.

In [ ]:
FEATS = ['n_recepciones','volumen_total_tn','ticket_medio_tn','merma_total_tn','humedad_media','humedad_std','humedad_max','pct_hum_crit','impurezas_media','impurezas_std','impurezas_max','pct_imp_crit','proteina_media','aceite_media','pct_descuento','max_merma_evento']
df = pd.read_csv('data/productores_features_v2.csv'); df.columns = df.columns.str.lower()
d = df[(df['id_productor'] != 525)].copy()
for c in FEATS: d[c] = pd.to_numeric(d[c], errors='coerce')
d = d.dropna(subset=FEATS).copy()
Xs = StandardScaler().fit_transform(np.log1p(d[FEATS].clip(lower=0)))
res = {}
for k in range(2, 8):
    lab = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(Xs)
    res[k] = (silhouette_score(Xs, lab), davies_bouldin_score(Xs, lab))
    print(f'K={k} sil={res[k][0]:.3f} DB={res[k][1]:.3f}')
K = max(res, key=lambda k: res[k][0])
final = KMeans(n_clusters=K, n_init=50, random_state=SEED).fit_predict(Xs)
ari = adjusted_rand_score(KMeans(n_clusters=K, n_init=20, random_state=1).fit_predict(Xs), KMeans(n_clusters=K, n_init=20, random_state=2).fit_predict(Xs))
print(f'K={K} ARI={ari:.3f}')
d['segmento'] = final
print(d.groupby('segmento')[FEATS].mean().round(2).to_string())

In [ ]:
nn = 2 * len(FEATS)
from sklearn.neighbors import NearestNeighbors
dist, _ = NearestNeighbors(n_neighbors=nn).fit(Xs).kneighbors(Xs)
db = DBSCAN(eps=float(np.quantile(np.sort(dist[:, -1]), 0.95)), min_samples=nn).fit(Xs)
print(f'DBSCAN atipicos: {(db.labels_ == -1).sum()}')
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(list(res), [res[k][0] for k in res], 'o-'); ax[0].set_title('Silueta vs K')
p = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
ax[1].scatter(p[:, 0], p[:, 1], c=final, cmap='tab10', s=10, alpha=.5); ax[1].set_title(f'K={K}')
d.groupby('segmento')['volumen_total_tn'].mean().plot.bar(ax=ax[2]); ax[2].set_title('Volumen medio')
plt.tight_layout(); plt.show()

## Frente 2 · Fraude (artefactos)
Score 0-100 + 5 tipologías + severidad por percentil. Fuente: `mineria_fraude_bascula.py` (7 cruces Bronce).

In [ ]:
assert os.path.exists('data/alerta_top.csv'), 'Falta data/alerta_top.csv: corre el script y commitea'
top = pd.read_csv('data/alerta_top.csv'); rs = pd.read_csv('data/riesgo_silo.csv'); rm = pd.read_csv('data/riesgo_mes.csv')
print('Alertas:', len(top), '| tipologias:', top['tipologia'].value_counts().to_dict())
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
top['tipologia'].value_counts().plot.bar(ax=ax[0], color='#b91c1c'); ax[0].set_title('Top por tipologia')
top['severidad'].value_counts().reindex(['bajo','medio','alto','critico']).plot.bar(ax=ax[1], color='#ea580c'); ax[1].set_title('Por severidad')
rs.set_index(rs.columns[0])['mean'].head(10).plot.barh(ax=ax[2]); ax[2].set_title('Top silos')
plt.tight_layout(); plt.show()
print('Meses criticos:'); print(rm.sort_values('max', ascending=False).head(3).to_string())

## Frente 3 · Regímenes IoT + CBOT (artefactos)
K=5 operativo, K=2 calmo/volátil. Fuente: `mineria_regimenes.py`.

In [ ]:
assert os.path.exists('data/mercado_regimen_hora.csv'), 'Falta data/mercado_regimen_hora.csv'
ri = pd.read_csv('data/iot_sensor_regimen.csv', names=['sensor_id', 'regimen'], header=0)
rm2 = pd.read_csv('data/mercado_regimen_hora.csv')
print('Sensores por regimen:'); print(ri['regimen'].value_counts().sort_index().to_string())
print(rm2.groupby('regimen')[['volatilidad', 'volumen', 'rango']].mean().round(4).to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ri['regimen'].value_counts().sort_index().plot.bar(ax=ax[0]); ax[0].set_title('Sensores por regimen')
rm2.groupby('regimen')['volatilidad'].mean().plot.bar(ax=ax[1]); ax[1].set_title('Volatilidad por regimen')
plt.tight_layout(); plt.show()

## Frente 4 · Asignación óptima (artefacto)
PuLP minimiza merma con capacidad. Medido: baseline 232.0 TN → óptimo 209.1 TN (**9.9%**, 22.9 TN/semana).

In [ ]:
assert os.path.exists('data/asignacion_optima.csv'), 'Falta data/asignacion_optima.csv'
a = pd.read_csv('data/asignacion_optima.csv')
print(f'Asignaciones: {len(a)} | silos usados: {a.iloc[:, 1].nunique()}')
a.iloc[:, 1].value_counts().head(10).plot.bar(figsize=(10, 3)); plt.title('Top silos cargados'); plt.show()
print('Ahorro medido en corrida full: 9.9% (22.9 TN/semana ≈ 1,190 TN/año). Supuestos: +20% sin control, +10% lleno>90%, capacidad nominal.')

## Cierre
**Segmentos** (base vs intensivos con picos, cuarentena 525) · **Fraude** (score + tipologías, percentil) · **Regímenes** (IoT K=5, CBOT K=2) · **Asignación** (9.9%).
**Despliegue:** `ORO.MINERIA_SEGMENTO_PRODUCTOR`, `ORO.MINERIA_ALERTA_BASCULA`, `ORO.MINERIA_REGIMEN_*` + re-ejecución por campaña. **Riesgos:** sintéticos (inventarios, fechas 1990/2099, contrato 1).